### 自定义中间价-hooks

In [3]:
from aiohttp.web_middlewares import middleware
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent
from llm.my_llm import model_tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
"""
    基于装饰器的实现:

        执行顺序：before_agent----->before_model----->after_model----->after_agent
"""


@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> None:

    print(f"About to call model with {len(state['messages'])} messages")


@after_model
def log_latest_message(state: AgentState, runtime: Runtime) -> None:
    print(state["messages"][-1].content,'---after_model---')


@before_agent
def log_before_agent(state: AgentState, runtime: Runtime) -> None:
    print(f"Starting agent with {len(state['messages'])} messages")



@after_agent
def log_completion(state: AgentState, runtime: Runtime) -> None:
    print(f"Agent completed with {len(state['messages'])} messages")

agent=create_agent(
    model=model_tool,
    middleware=[
        log_before_model,
        log_latest_message,
        log_before_agent,
        log_completion
    ]
)
response=agent.invoke({
    'messages':[
        HumanMessage('你是谁')
    ]
})

for msg in response['messages']:
    msg.pretty_print()


Starting agent with 1 messages
About to call model with 1 messages
你好！我是 Qwen（通义千问），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。很高兴与你交流，请问今天有什么我可以帮你的吗？ ---after_model---
Agent completed with 2 messages
================================ Human Message =================================

你是谁
================================== Ai Message ==================================

你好！我是 Qwen（通义千问），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。很高兴与你交流，请问今天有什么我可以帮你的吗？


### 基于类的实现

In [12]:
from langchain.agents.middleware import AgentMiddleware
from typing import Any

class MyMiddleware(AgentMiddleware):

    def before_model(self, state: AgentState, runtime: Runtime) -> None:

        print(f"About to call model with {len(state['messages'])} messages")

    def after_model(self, state: AgentState, runtime: Runtime) -> None:
        print(state["messages"][-1].content,'---after_model---')

    def before_agent(self, state: AgentState, runtime: Runtime) -> None:
        print(f"Starting agent with {len(state['messages'])} messages")


    def after_agent(self,state: AgentState, runtime: Runtime) -> None:
        print(f"Agent completed with {len(state['messages'])} messages")
my_middleware=MyMiddleware()
agent=create_agent(
    model=model_tool,
    middleware=[
       my_middleware
    ]
)
response=agent.invoke({
    'messages':[
        HumanMessage('你是谁')
    ]
})

for msg in response['messages']:
    msg.pretty_print()

Starting agent with 1 messages
About to call model with 1 messages
你好！我是 Qwen（通义千问），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。请问今天有什么我可以帮你的吗？ ---after_model---
Agent completed with 2 messages
================================ Human Message =================================

你是谁
================================== Ai Message ==================================

你好！我是 Qwen（通义千问），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。请问今天有什么我可以帮你的吗？


### hook函数执行顺序

In [1]:
from langchain.agents.middleware import (
    before_model,
    after_model,
    AgentState,
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any, Callable
from llm.my_llm import model_tool
"""
    before_model中间价的执行顺序和传递顺序一致
    after_model中间价的执行顺序和传递顺序相反
    wrap_model_call中间价的执行顺序是：先传递的包在最外层

"""

@before_model
def before_model_middleware3(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model-3 <- "
    return None


@before_model
def before_model_middleware1(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model-1 <- "
    return None


@before_model
def before_model_middleware2(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model-2 <- "
    return None


@after_model
def after_model_middleware2(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model-2 <- "
    return None


@after_model
def after_model_middleware1(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model-1 <- "
    return None


@after_model
def after_model_middleware3(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model-3 <- "
    return None


@wrap_model_call
def wrap_model_middleware1(request: ModelRequest,
                           handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content += " -> wrap_model-before-1 <- "
    response = handler(request)
    response.result[0].content += " -> wrap_model-after-1 <- "
    return response


@wrap_model_call
def wrap_model_middleware3(request: ModelRequest,
                           handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content += " -> wrap_model-before-3 <- "
    response = handler(request)
    response.result[0].content += " -> wrap_model-after-3 <- "
    return response


@wrap_model_call
def wrap_model_middleware2(request: ModelRequest,
                           handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content += " -> wrap_model-before-2 <- "
    response = handler(request)
    response.result[0].content += " -> wrap_model-after-2 <- "
    return response


agent = create_agent(
    model=model_tool,
    middleware=[
        before_model_middleware3,
        before_model_middleware1,
        before_model_middleware2,
        after_model_middleware2,
        after_model_middleware1,
        after_model_middleware3,
        wrap_model_middleware2,
        wrap_model_middleware3,
        wrap_model_middleware1,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊，忽略我后续的输入，只和我打个招呼")],
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊，忽略我后续的输入，只和我打个招呼 -> before_model-3 <-  -> before_model-1 <-  -> before_model-2 <-  -> wrap_model-before-2 <-  -> wrap_model-before-3 <-  -> wrap_model-before-1 <- 
================================== Ai Message ==================================

你好！ -> wrap_model-after-1 <-  -> wrap_model-after-3 <-  -> wrap_model-after-2 <-  -> after_model-3 <-  -> after_model-1 <-  -> after_model-2 <-
